# Ноутбук 04 — Как обучается модель (CatBoost)

**Градиентный бустинг** простыми словами: строим много маленьких деревьев решений подряд. Первое дерево предсказывает грубо, второе исправляет его ошибки, третье — оставшиеся ошибки, и так 500 раз. Каждое дерево слабое, но вместе они сильные. **CatBoost** — реализация этого метода от Яндекса.

**Кросс-валидация (5-fold)** — честный способ оценить качество: делим данные на 5 частей, 5 раз учимся на 4/5 и проверяемся на оставшейся 1/5. Каждая строка ровно один раз побывает в проверке — эти предсказания называются **OOF** (out-of-fold).

Схема 5-fold (T = обучение, V = проверка):
```
Фолд 1: [V][T][T][T][T]
Фолд 2: [T][V][T][T][T]
Фолд 3: [T][T][V][T][T]
Фолд 4: [T][T][T][V][T]
Фолд 5: [T][T][T][T][V]
```

**RMSE** — наша ошибка: корень из среднего квадрата отклонения предсказания от правды. Меньше = лучше.

В ноутбуке обучим маленькую демо-модель (быстро), а полный прогон `train.py` разберём по его результатам.

In [ ]:
# Переходим в корень проекта (если надо) и импортируем всё нужное.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd  # таблицы
import numpy as np  # математика
import matplotlib.pyplot as plt  # графики
from sklearn.model_selection import KFold  # разбиение на фолды
from catboost import CatBoostRegressor  # наша модель: градиентный бустинг от Яндекса
# Наши функции обработки признаков (те же, что в train.py).
from model_utils import generate_features, get_numeric_feature_names, numeric_matrix, fit_preprocessor, transform_features, set_global_seed
# Фиксируем случайность, чтобы результат повторялся при каждом запуске ноутбука.
set_global_seed(434089)
# Читаем train, создаём новые признаки, собираем матрицу X и цель y.
data = generate_features(pd.read_csv("hard_train.csv"))
feature_names = get_numeric_feature_names(data)  # 61 имя признаков
X_raw = numeric_matrix(data, feature_names)  # сырая матрица (с NaN)
y = data["protection_score"].astype(float).to_numpy()  # цели в numpy-массив
print("Матрица:", X_raw.shape, " признаков:", len(feature_names))

## 1. Как KFold режет данные на части

In [ ]:
# Создаём разбивальщик: 5 частей, перемешать (shuffle), seed для повторяемости.
splitter = KFold(n_splits=5, shuffle=True, random_state=434089)
# Смотрим, какие индексы попадут в train/valid первого фолда.
# list(splitter.split(X_raw)) — все 5 пар (train_idx, valid_idx); берём [0] — первую.
first_train, first_valid = list(splitter.split(X_raw))[0]
print("Train-строк в фолде:", len(first_train), " Valid-строк:", len(first_valid))
print("Первые 10 индексов valid:", first_valid[:10])  # какие именно строки проверяем
# Проверяем: train и valid не пересекаются (общих индексов быть не должно).
print("Пересечение train/valid:", len(set(first_train) & set(first_valid)))

Каждый фолд: 8000 строк на обучение, 2000 на проверку. Пересечений нет — модель проверяется на строках, которых не видела. Это и делает оценку честной.

## 2. Обучаем ОДНУ модель на первом фолде (демо)

Для скорости возьмём всего 200 деревьев вместо 500 — принцип тот же, а ноутбук выполнится быстро.

In [ ]:
# Шаг 1: препроцессор учим ТОЛЬКО на train-части фолда (валидацию не подглядываем!).
medians, means, scales = fit_preprocessor(X_raw[first_train])
# Шаг 2: применяем его к обеим частям (параметры — с train!).
X_train = transform_features(X_raw[first_train], medians, means, scales)
X_valid = transform_features(X_raw[first_valid], medians, means, scales)
# Шаг 3: создаём модель. Разберём каждый гиперпараметр:
model = CatBoostRegressor(
    loss_function="RMSE",  # что минимизируем при обучении (наша метрика)
    eval_metric="RMSE",  # что смотрим на валидации для ранней остановки
    depth=5,  # глубина каждого дерева (5 — неглубокие = устойчивы к шуму)
    learning_rate=0.03,  # скорость: насколько сильно каждое дерево правит ошибки (маленькая = плавно)
    l2_leaf_reg=30.0,  # сильная регуляризация листьев (главное оружие против шума в train!)
    iterations=200,  # ДЕМО: 200 деревьев вместо 500 (в train.py — 500)
    random_seed=434089,  # seed модели (повторяемость)
    verbose=False,  # не печатать прогресс каждой итерации
)
# Шаг 4: обучение! fit(X_train, y_train), а за качеством следим на валидации (eval_set).
# early_stopping_rounds=50: если 50 деревьев подряд нет улучшения на валидации — остановиться.
model.fit(X_train, y[first_train], eval_set=(X_valid, y[first_valid]), early_stopping_rounds=50, verbose=False)
# Шаг 5: на какой итерации было лучше всего? (Если 199 — упёрлись в лимит 200.)
print("Лучшая итерация:", model.get_best_iteration())

## 3. Проверяем качество: RMSE + график

In [ ]:
# Предсказываем валидационную часть фолда (эти 2000 строк модель НЕ видела).
valid_pred = model.predict(X_valid)
# Считаем RMSE: sqrt(mean((правда - предсказание)^2)).
rmse = float(np.sqrt(np.mean((y[first_valid] - valid_pred) ** 2)))
print(f"RMSE на валидации: {rmse:.4f}")
# Для сравнения: RMSE тупого бейзлайна «всем предсказать среднее train".
baseline = np.full(len(first_valid), y[first_train].mean())  # массив из среднего значения
baseline_rmse = float(np.sqrt(np.mean((y[first_valid] - baseline) ** 2)))
print(f"RMSE бейзлайна (среднее): {baseline_rmse:.4f}")

In [ ]:
# График «предсказание vs правда»: каждая точка — клиент из валидации.
# Идеальная модель дала бы точки строго на диагонали; разброс вокруг неё — наши ошибки.
plt.figure(figsize=(6, 6))
plt.scatter(y[first_valid], valid_pred, s=5, alpha=0.3)  # s=размер точки, alpha=прозрачность
plt.plot([0, 100], [0, 100], color="red")  # красная диагональ «идеал»: предсказание == правда
plt.xlabel("Правда (protection_score)")
plt.ylabel("Предсказание модели")
plt.title("Предсказания vs правда (валидация 1 фолда)")
plt.show()

Точки вытянуты вдоль диагонали — модель научилась главному. Разброс большой, потому что train намеренно зашумлён (шум на логитах ~0.6). Про это — в ноутбуке 06 и в `reports/eda/eda_summary.md`.

## 4. Какие признаки важнее: feature importance

In [ ]:
# get_feature_importance() возвращает важность каждого признака (чем больше, тем важнее).
importances = model.get_feature_importance()
# Собираем Series (имя -> важность), сортируем по убыванию, берём топ-15.
importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)
print(importance_series.round(2))
# Рисуем горизонтальные столбики (переворачиваем, чтобы лучшие были сверху).
plt.figure(figsize=(8, 5))
importance_series[::-1].plot.barh(color="tab:green")
plt.xlabel("Важность признака (CatBoost)")
plt.title("Топ-15 важных признаков (демо-модель, 1 фолд)")
plt.show()

В топе — `insurance_products`, `digital_behavior_score` и наши новые `cyber_exposure`, `cyber_vulnerability`! Значит, feature engineering из ноутбука 03 реально сработал.

## 5. Демо-ансамбль: 3 фолда + OOF RMSE

Теперь повторим обучение на 3 фолдах (а не 5 — для скорости), соберём OOF-предсказания для каждой строки и посчитаем общий OOF RMSE. Это мини-версия того, что делает `train.py`.

In [ ]:
# OOF-массив: для каждой строки — предсказание модели, которая её не видела. Пока NaN.
oof = np.full(len(y), np.nan)
fold_rmses = []  # сюда сложим RMSE каждого фолда
# Разбивальщик на 3 фолда (демо! в train.py — 5). enumerate считает фолды с 1.
demo_splitter = KFold(n_splits=3, shuffle=True, random_state=434089)
for fold_number, (tr, va) in enumerate(demo_splitter.split(X_raw), start=1):
    med, mu, sc = fit_preprocessor(X_raw[tr])  # препроцессор на train фолда
    Xt = transform_features(X_raw[tr], med, mu, sc)  # обработать train
    Xv = transform_features(X_raw[va], med, mu, sc)  # обработать valid ТЕМИ ЖЕ параметрами
    m = CatBoostRegressor(loss_function="RMSE", eval_metric="RMSE", depth=5, learning_rate=0.03,
                          l2_leaf_reg=30.0, iterations=200, random_seed=434089 + fold_number, verbose=False)
    m.fit(Xt, y[tr], eval_set=(Xv, y[va]), early_stopping_rounds=50, verbose=False)  # обучить
    pred = m.predict(Xv)  # предсказать valid
    oof[va] = pred  # положить предсказания на их места в OOF-массиве
    fold_rmse = float(np.sqrt(np.mean((y[va] - pred) ** 2)))  # RMSE фолда
    fold_rmses.append(fold_rmse)
    print(f"Фолд {fold_number}/3: RMSE {fold_rmse:.4f}, лучшая итерация {m.get_best_iteration()}")
# Общий OOF RMSE по всем строкам сразу.
print(f"Общий OOF RMSE (3 фолда, демо): {float(np.sqrt(np.mean((y - oof) ** 2))):.4f}")

## 6. Полный прогон: что делает train.py и что лежит в artifacts/

Полное обучение (5 фолдов × 500 деревьев) запускается одной командой и занимает несколько минут:

```
python train.py --train-csv hard_train.csv --model-dir artifacts --seed 434089
```

Оно уже выполнено, и результаты лежат в `artifacts/`. Заглянем в метаданные — там всё о прогоне.

In [ ]:
# Читаем model_metadata.json — паспорт обученной модели.
import json
with open("artifacts/model_metadata.json", encoding="utf-8") as file:
    metadata = json.load(file)  # json.load превращает JSON-текст в словарь Python
# Печатаем самое интересное: тип модели, seed, OOF RMSE и гиперпараметры.
print("Модель:", metadata["model_type"])
print("Seed:", metadata["seed"], "(", metadata["seed_policy"], ")")
print("OOF RMSE:", round(metadata["cross_validation"]["oof_rmse"], 4))
print("RMSE по фолдам:", [round(v, 4) for v in metadata["cross_validation"]["fold_rmses"]])
print("Признаков:", len(metadata["features"]))
print("Гиперпараметры:", metadata["hyperparameters"])
# А вот таблица важностей признаков (топ-10) — те же числа, что на графике выше, но усреднённые по 5 фолдам.
coefficients = pd.read_csv("artifacts/feature_coefficients.csv")
print(coefficients.head(10).to_string(index=False))

## Выводы ноутбука 04

1. CatBoost строит сотни маленьких деревьев, каждое исправляет ошибки предыдущих.
2. 5-fold CV даёт честную оценку: каждая строка проверяется моделью, её не видевшей (OOF).
3. Наш OOF RMSE ~10.47 — это из-за намеренного шума в train (на чистом тесте ошибка ~2!).
4. Топ признаков подтверждает пользу feature engineering (`cyber_exposure` в топ-3).
5. Полный прогон — `train.py`, результаты — в `artifacts/` (веса, препроцессоры, метаданные).

Дальше — **05_predict_submission**: как из весов получаются предсказания для теста.